## [Summarization](https://huggingface.co/course/chapter7/5?fw=pt)

In this section we'll take a look at how Transformer models can be used to condense long documents into summaries, a task known as *text summarization*. This is one of the most challenging NLP tasks as it requires a range of abilities, such as understanding long passages and generating coherent text that captures the main topics in a document. However, when done well, text summarization is a powerful tool that can speed up various business processes by relieving the burden of domain experts to read long documents in detail.

In [1]:
import os
from IPython.display import HTML
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/yHnr5Dk2zCI" allowfullscreen></iframe>')

/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/IPython/core/display.py:475: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Although there already exist various fine-tuned models for summarization on the [Hugging Face Hub](https://huggingface.co/models?pipeline_tag=summarization&sort=downloads), almost all of these are only suitable for English documents. By the end of this section, you'll have a [model](https://huggingface.co/huggingface-course/mt5-small-finetuned-amazon-en-es) that can summarize customer reviews like the one shown here:

> Simply put, AMD's flagship Ryzen 9 7950X is a multi-threading monster of a mainstream CPU. It commands a hefty power draw, an even heftier cooler, and (most consequential to you) a steep price, at 699 USD MSRP (though discounted a bit these days). For certain professions in which time literally is money, increased performance means the increased output of projects. However, if you have high-tech demands but your livelihood isn't determined by your PC's processing power, models just under the flagship often deliver nearly as much performance but are available for significantly less, like the 549 USD MSRP AMD Ryzen 9 7900X [[PCmag](https://uk.pcmag.com/processors/146625/amd-ryzen-9-7900x)].

### Preparing a corpus
We'll use the [buruzaemon/amazon_reviews_multi](https://huggingface.co/datasets/buruzaemon/amazon_reviews_multi) dataset to create our summarizer. To get started, let's download the dataset from the Hugging Face Hub and only keep reviews that were written in English or German.

In [45]:
from datasets import load_dataset
amazon_reviews_dataset = (
    load_dataset("buruzaemon/amazon_reviews_multi")
    .remove_columns(["product_id", "reviewer_id", "review_id", "stars", "product_category"])
)
#amazon_reviews_dataset_EnDe = amazon_reviews_dataset.filter(lambda item: item["language"]=="en" or "de")
language_set = {"en", "de"}
amazon_reviews_dataset_EnDe = amazon_reviews_dataset.filter(lambda item: item["language"] in language_set)
print(f"language_set = {language_set}")
assert set(amazon_reviews_dataset_EnDe["test"]["language"]) == language_set
amazon_reviews_dataset_EnDe

language_set = {'en', 'de'}


DatasetDict({
    train: Dataset({
        features: ['review_body', 'review_title', 'language'],
        num_rows: 400000
    })
    validation: Dataset({
        features: ['review_body', 'review_title', 'language'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['review_body', 'review_title', 'language'],
        num_rows: 10000
    })
})

As you can see after filtering out all languages but English and German, there are `400000` reviews for the `train` split, `10000` reviews for the `validation` split, and also `10000` reviews for `test split`. The review information we are interested in are contained in the `review_body` (original review) and `review_title` (review summary) columns. Let's take a look at a few examples by creating a simple function that takes a random sample from the training set with the techniques we learned in Chapter 5:

In [46]:
def show_samples(amazon_reviews_dataset_EnDe, num_samples=5, seed=42):
    sample = amazon_reviews_dataset_EnDe["train"].shuffle(seed=seed).select(range(num_samples))
    for example in sample:
        print(f"\n>> Document: {example['review_body']}")
        print(f">> Summary: {example['review_title']}")
    pass
    
show_samples(amazon_reviews_dataset_EnDe)


>> Document: Fast delivery - received in 1 day. Package contains easy to follow user guide. The dash cam screen is about 2 inch by 1.5 inch, it is very easy to install. Video quality is great. Great quality, reasonable price. Definitely recommend it
>> Summary: Easy to use, great quality

>> Document: It’s a perfect on a key chain and the LEDs are super bright too
>> Summary: Very pleased, comes as it looks

>> Document: Fits well! I haven't noticed much shrinking. It's super comfy, and does not slide or roll (as they do sometimes on bigger ladies).
>> Summary: It's super comfy, and does not slide or roll (as ...

>> Document: Diese Feuerschale ist top. Sehr einfach aufzubauen, die Beine lassen sich recht simpel und platzsparend einklappen. Der Deckel läßt den Blick auf das Feuer gut durch und grobe Stücke nicht durch.
>> Summary: Super Teil für unsere Terrasse

>> Document: Not my kids favorite. Seemed to only like it with watermelon
>> Summary: Good


> ✏️ **Try it out!** <font color="darkgreen">Change the random seed in the `Dataset.shuffle()` command to explore other reviews in the corpus.
</font>

In [74]:
from random import randrange
seed = randrange(100)
# try the following seeds: 1 (two German texts)
#                          2 (a German text and an English text)
#                          3 (two English texts)
seed = 2
print(f"seed = {seed}")
def show_samples(amazon_reviews_dataset_EnDe, num_samples=2):
    sample = amazon_reviews_dataset_EnDe["train"].shuffle(seed).select(range(num_samples))
    for example in sample:
        print(f"\n>> Document: {example['review_body']}")
        print(f">> Summary: {example['review_title']}")
show_samples(amazon_reviews_dataset_EnDe)

seed = 2

>> Document: Super Preis-Leistungsverhältnis. Habe diese für eine größere Jugendgruppe bestellt. Die Schläger sind zwar nicht die besten, entsprechend dem Preis aber tun was sie sollen. Die Bälle hätte ich etwas besser vorgestellt von der Qualität her.
>> Summary: Gut!

>> Document: We received the product- Jack black today and the packaging was torn and open.
>> Summary: Jack Black


With `seed = 1`, both texts are in German.<br>The **first** text is about a superficial comic or book that doesn't make a lot of sense.
According to the ***summary***, the buyer doesn't appreciate the product.<br>The **second** product seems to be about batteries that last very long and have a lot of juice. The customer would buy them again and can really recommend them.
According to the ***summary***, the customer finds that these batteries are awesome.

With `seed = 2`, the first text is in German and the second text is in English.<br>The **first** and German text seems to be about tennis or table tennis equipment. The rackets and balls don't seem to be of the highest quality but seem to be a fair deal given the (seemingly low) price. The ***summary*** simply reads "Gut!" ("Good!")<br>The **second** and English text is about a "Jack black" product that has arrived "today", yet with a torn and open packaging.
The ***summary*** simply reads "Jack Black".

With `seed = 3`, both texts are in English.<br>The **first** text is about product with a wet/dry stainless steel filter. Apparently, there was some auction but the participants didn't value that filter fairly. As a consequnce, the ride share driver who offered the product was a bit disappointed with the auction.
According to the ***summary***, the products lack suction power.<br>The **second** text mentions that "tracking shows" that the carrier didn't leave the product in the mailbox.<br>Accordingly, the ***summary*** states "Didn't receive item".

This certainly looks like a mix of English and Spanish reviews!

Now that we have a training corpus, one final thing to check is the distribution of words in the reviews and their titles. This is especially important for summarization tasks, where short reference summaries in the data can bias the model to only output one or two words in the generated summaries. The plots below show the word distributions, and we can see that the titles are heavily skewed toward just 1-2 words:
Word count distributions for the review titles and texts.

<img style="float=center;" src="sections/section_7/images/Review_title_and_body.png" width="80%">

To deal with this, we'll filter out the examples with very short titles so that our model can produce more interesting summaries. Since we're dealing with English and German texts, we can use a rough heuristic to split the titles on whitespace and then use our trusty `Dataset.filter()` method as follows:

In [82]:
amazon_reviews_dataset_EnDe = amazon_reviews_dataset_EnDe.filter(lambda x: x is not None)                 # dataset row isn't None
amazon_reviews_dataset_EnDe = amazon_reviews_dataset_EnDe.filter(lambda x: x["review_title"] is not None) # row's review_title isn't None
amazon_reviews_dataset_EnDe = (                                                                           # only keep rows where ...
    amazon_reviews_dataset_EnDe                                                                           # ... "review_title" has ...
    .filter(lambda x: len(x["review_title"].split()) >= 3)                                                # ... 3 words or more
)
amazon_reviews_dataset_EnDe

Filter:   0%|          | 0/233043 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5778 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5836 [00:00<?, ? examples/s]

Filter:   0%|          | 0/233043 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5778 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5836 [00:00<?, ? examples/s]

Filter:   0%|          | 0/233043 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5778 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5836 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['review_body', 'review_title', 'language'],
        num_rows: 233043
    })
    validation: Dataset({
        features: ['review_body', 'review_title', 'language'],
        num_rows: 5778
    })
    test: Dataset({
        features: ['review_body', 'review_title', 'language'],
        num_rows: 5836
    })
})

Now that we've prepared our corpus, let's take a look at a few possible Transformer models that one might fine-tune on it!

### Models for text summarization
If you think about it, text summarization is a similar sort of task to machine translation: we have a body of text like a review that we'd like to "translate" into a shorter version that captures the salient features of the input. Accordingly, most Transformer models for summarization adopt the encoder-decoder architecture that we first encountered in Chapter 1, although there are some exceptions like the GPT family of models which can also be used for summarization in few-shot settings. The following table lists some popular pretrained models that can be fine-tuned for summarization.

|Transformer model|Description|Multilingual?|
|:---|:---|:---:|
|[GPT-2](https://huggingface.co/gpt2-xl)|Although trained as an auto-regressive language model, you can make GPT-2 generate summaries by appending "TL;DR" at the end of the input text.|❌|
|[PEGASUS](https://huggingface.co/google/pegasus-large)|Uses a pretraining objective to predict masked sentences in multi-sentence texts. This pretraining objective is closer to summarization than vanilla language modeling and scores highly on popular benchmarks.|❌|
|[T5](https://huggingface.co/t5-base)|A universal Transformer architecture that formulates all tasks in a text-to-text framework; e.g., the input format for the model to summarize a document is `summarize: ARTICLE`.|❌|
|[mT5](https://huggingface.co/google/mt5-base)|A multilingual version of T5, pretrained on the multilingual Common Crawl corpus (mC4), covering 101 languages.|✅|
|[BART](https://huggingface.co/facebook/bart-base)|A novel Transformer architecture with both an encoder and a decoder stack trained to reconstruct corrupted input that combines the pretraining schemes of BERT and GPT-2.|❌|
|[mBART-50](https://huggingface.co/facebook/mbart-large-50)|A multilingual version of BART, pretrained on 50 languages.|✅|

As you can see from this table, the majority of Transformer models for summarization (and indeed most NLP tasks) are monolingual. This is great if your task is in a "high-resource" language like English or German, but less so for the thousands of other languages in use across the world. Fortunately, there is a class of multilingual Transformer models, like mT5 and mBART, that come to the rescue. These models are pretrained using language modeling, but with a twist: instead of training on a corpus of one language, they are trained jointly on texts in over 50 languages at once!

We'll focus on mT5, an interesting architecture based on T5 that was pretrained in a text-to-text framework. In T5, every NLP task is formulated in terms of a prompt prefix like `summarize:` which conditions the model to adapt the generated text to the prompt. As shown in the figure below, this makes T5 extremely versatile, as you can solve many tasks with a single model!

<img style="float=center;" src="sections/section_7/images/T5_versatility.png" width="80%">

mT5 doesn't use prefixes, but shares much of the versatility of T5 and has the advantage of being multilingual. Now that we've picked a model, let's take a look at preparing our data for training.

> ✏️ **Try it out!** <font color="darkgreen">Once you've worked through this section, see how well mT5 compares to mBART by fine-tuning the latter with the same techniques. For bonus points, you can also try fine-tuning T5 on just the English reviews. Since T5 has a special prefix prompt, you'll need to prepend `summarize:` to the input examples in the preprocessing steps below.</font>

### Preprocessing the data

In [83]:
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/1m7BerpSq8A" allowfullscreen></iframe>')

/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/IPython/core/display.py:475: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Our next task is to tokenize and encode our reviews and their titles. As usual, we begin by loading the tokenizer associated with the pretrained model checkpoint. We'll use `mt5-small` as our checkpoint so we can fine-tune the model in a reasonable amount of time:

In [84]:
from transformers import AutoTokenizer
model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warn

T5TokenizerFast(name_or_path='google/mt5-small', vocab_size=250100, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	250000: AddedToken("▁<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	250001: AddedToken("▁<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	250002: AddedToken("▁<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	250003: Add

> <font color="darkgreen">💡 In the early stages of your NLP projects, a good practice is to train a class of "small" models on a small sample of data. This allows you to debug and iterate faster toward an end-to-end workflow. Once you are confident in the results, you can always scale up the model by simply changing the model checkpoint!</font>

Let's test out the mT5 tokenizer on a small example:

In [85]:
inputs = tokenizer("I loved reading the Hunger Games!")
inputs

{'input_ids': [336, 259, 28387, 11807, 287, 62893, 295, 12507, 309, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Here we can see the familiar `input_ids` and `attention_mask` that we encountered in our first fine-tuning experiments back in [Chapter 3](https://huggingface.co/course/chapter3). Let's decode these input IDs with the tokenizer's `convert_ids_to_tokens()` function to see what kind of tokenizer we're dealing with:

In [86]:
tokenizer.convert_ids_to_tokens(inputs.input_ids)

['▁I', '▁', 'loved', '▁reading', '▁the', '▁Hung', 'er', '▁Games', '!', '</s>']

The special Unicode character `▁` and end-of-sequence token `</s>` indicate that we’re dealing with the SentencePiece tokenizer, which is based on the Unigram segmentation algorithm discussed in [Chapter 6](https://huggingface.co/course/chapter6). Unigram is especially useful for multilingual corpora since it allows SentencePiece to be agnostic about accents, punctuation, and the fact that many languages, like Japanese, do not have whitespace characters.

To tokenize our corpus, we have to deal with a subtlety associated with summarization: because our labels are also text, it is possible that they exceed the model's maximum context size. This means we need to apply truncation to both the reviews and their titles to ensure we don't pass excessively long inputs to our model. The tokenizers in 🤗 Transformers provide a nifty `text_target` argument that allows you to tokenize the labels in parallel to the inputs. Here is an example of how the inputs and targets are processed for mT5:

In [87]:
max_input_length = 512
max_target_length = 30

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["review_body"],
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        examples["review_title"], max_length=max_target_length, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

Let's walk through this code to understand what's happening. The first thing we've done is define values for `max_input_length` and `max_target_length`, which set the upper limits for how long our reviews and titles can be. Since the review body is typically much larger than the title, we've scaled these values accordingly.

With `preprocess_function()`, it is then a simple matter to tokenize the whole corpus using the handy `Dataset.map()` function we've used extensively throughout this course:

In [90]:
tokenized_datasets = amazon_reviews_dataset_EnDe.map(preprocess_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/233043 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['review_body', 'review_title', 'language', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 233043
    })
    validation: Dataset({
        features: ['review_body', 'review_title', 'language', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 5778
    })
    test: Dataset({
        features: ['review_body', 'review_title', 'language', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 5836
    })
})

Now that the corpus has been preprocessed, let's take a look at some metrics that are commonly used for summarization. As we'll see, there is no silver bullet when it comes to measuring the quality of machine-generated text.

> <font color="darkgreen">💡 You may have noticed that we used `batched=True` in our `Dataset.map()` function above. This encodes the examples in batches of 1000 (the default) and allows you to make use of the multithreading capabilities of the fast tokenizers in 🤗 Transformers. Where possible, try using `batched=True` to get the most out of your preprocessing!</font>

### Metrics for text summarization

In [92]:
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/TMshhnrEXlg" allowfullscreen></iframe>')

In comparison to most of the other tasks we've covered in this course, measuring the performance of text generation tasks like summarization or translation is not as straightforward. For example, given a review like "I loved reading the Hunger Games", there are multiple valid summaries, like "I loved the Hunger Games" or "Hunger Games is a great read". Clearly, applying some sort of exact match between the generated summary and the label is not a good solution — even humans would fare poorly under such a metric, because we all have our own writing style.

For summarization, one of the most commonly used metrics is the [ROUGE score](https://en.wikipedia.org/wiki/ROUGE_(metric)) (short for Recall-Oriented Understudy for Gisting Evaluation). The basic idea behind this metric is to compare a generated summary against a set of reference summaries that are typically created by humans. To make this more precise, suppose we want to compare the following two summaries:

In [93]:
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

One way to compare them could be to count the number of overlapping words, which in this case would be 6. However, this is a bit crude, so instead ROUGE is based on computing the *precision* and *recall* scores for the overlap.

> <font color="darkgreen">🙋 Don't worry if this is the first time you've heard of precision and recall — we'll go through some explicit examples together to make it all clear. These metrics are usually encountered in classification tasks, so if you want to understand how precision and recall are defined in that context, we recommend checking out the `scikit-learn` [guides](https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html).</font>

For ROUGE, recall measures how much of the reference summary is captured by the generated one. If we are just comparing words, recall can be calculated according to the following formula: 
$$\text{Recall}=\frac{\text{Number of overlapping words}}{\text{Total number of words in reference summary}}$$
Applying this to our verbose summary gives a precision of 6/10 = 0.6, which is considerably worse than the precision of 6/7 = 0.86 obtained by our shorter one. In practice, both precision and recall are usually computed, and then the $F_1$-score (the harmonic mean of precision and recall) is reported. We can do this easily in 🤗 Datasets by first installing the `rouge_score` package:

In [95]:
# The following command needs to run only once.
# !pip install rouge_score

and then loading the ROUGE metric as follows:

In [97]:
import evaluate
rouge_score = evaluate.load("rouge")

Then we can use the `rouge_score.compute()` function to calculate all the metrics at once:

In [98]:
scores = rouge_score.compute(predictions=[generated_summary], references=[reference_summary])
scores

{'rouge1': np.float64(0.923076923076923),
 'rouge2': np.float64(0.7272727272727272),
 'rougeL': np.float64(0.923076923076923),
 'rougeLsum': np.float64(0.923076923076923)}

Whoa — what do these values mean? 🤗 Datasets computes a variety of ROUGE scores which are based on different types of text granularity when comparing the generated and reference summaries.
- The `rouge1` variant is the overlap of unigrams — this is just a fancy way of saying the overlap of words and is exactly the metric we've discussed above.
- The `rouge2` variant is the overlap of bigrams — this is just a fancy way of saying the overlap of two-word sequences.
- The `rougeL` variant only cares about order, not adjacency: `"the cat sat on the mat"` and `"the dog sat on mat"` both have `the`, `sat`, `on`, and `mat` in the same order, irrespective of "interrupting" words in between.
- The `rougeLsum` variant measures the longest sequence of words shared in order, irrespective of "interrupting" words, between the reference and generated summaries — across full texts, not just sentence pairs — making it well-suited for evaluating summarization quality.

> ✏️ **Try it out!** <font color="darkgreen">Create your own example of a generated and reference summary and see if the resulting ROUGE scores agree with a manual calculation based on the formulas for precision and recall. For bonus points, split the text into bigrams and compare the precision and recall for the `rouge2` metric.</font>

We'll use these ROUGE scores to track the performance of our model, but before doing that let's do something every good NLP practitioner should do: create a strong, yet simple baseline!

### Creating a strong baseline
A common baseline for text summarization is to simply take the first three sentences of an article, often called the *lead-3* baseline. We could use full stops to track the sentence boundaries, but this will fail on acronyms like "U.S." or "U.N." — so instead we'll use the `nltk` library, which includes a better algorithm to handle these cases. You can install the package using `pip` as follows:

In [104]:
# this command needs to run only once
#!pip install nltk

And then download the punctuation rules:

In [108]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /home/matthias/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/matthias/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

Next, we import the sentence tokenizer from `nltk` and create a simple function to extract the first three sentences in a review. The convention in text summarization is to separate each summary with a newline, so let's also include this and test it on a training example:

In [111]:
from nltk.tokenize import sent_tokenize
def three_sentence_summary(text):
    return "\n".join(sent_tokenize(text)[:3])

print(three_sentence_summary(amazon_reviews_dataset_EnDe["train"][0]["review_body"]))

Armband ist leider nach 1 Jahr kaputt gegangen


This seems to work, so let's now implement a function that extracts these "summaries" from a dataset and computes the ROUGE scores for the baseline:

In [112]:
def evaluate_baseline(dataset, metric):
    summaries = [three_sentence_summary(text) for text in dataset["review_body"]]
    return metric.compute(predictions=summaries, references=dataset["review_title"])

We can then use this function to compute the ROUGE scores over the validation set and prettify them a bit using `pandas`:

In [116]:
import pandas as pd
score = evaluate_baseline(amazon_reviews_dataset_EnDe["validation"], rouge_score)
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = dict((rn, round(score[rn].mid.fmeasure * 100, 2)) for rn in rouge_names)
rouge_dict

AttributeError: 'numpy.float64' object has no attribute 'mid'